In [2]:
# Instalar Selenium, Scrapy y dependencias para Chrome en el entorno Linux de Colab
!pip install selenium scrapy beautifulsoup4
!apt-get update
!apt-get install -y chromium-browser chromium-chromedriver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 42.0 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.9/885.9 kB 37.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.6 MB/s  0:00:00
  Attempting uninstall: urllib3━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/31 [zope-interface]
    Found existing installation: urllib3 2.5.0━━━━━━━━━━━━━━━━  1/31 [zope-interface]
    Uninstalling urllib3-2.5.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/31 [zope-interface]
      Successfully uninstalled urllib3-2.5.0━━━━━━━━━━━━━━━━━━━━━━  3/31 [urllib3]ce]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31/31 [scrapy]30/31 [scrapy]m]dentity]
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]      
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]       
Hit:5 http://archive

In [4]:
from google.colab import drive
import os

# Montar Drive para guardar los datos finales
drive.mount('/content/drive')

# Crear carpeta de trabajo si no existe
carpeta_destino = '/content/drive/MyDrive/p-analisis_datos-g10'
os.makedirs(carpeta_destino, exist_ok=True)
print(f"Carpeta lista en: {carpeta_destino}")

Mounted at /content/drive
Carpeta lista en: /content/drive/MyDrive/p-analisis_datos-g10


In [5]:
import requests
from bs4 import BeautifulSoup

url_static = "http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
headers = {'User-Agent': 'Mozilla/5.0'}

response = requests.get(url_static, headers=headers)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    titulo = soup.find('h1').text
    precio = soup.find('p', class_='price_color').text
    disponibilidad = soup.find('p', class_='instock availability').text.strip()

    datos_bs4 = {
        "metodo": "BeautifulSoup",
        "titulo": titulo,
        "precio": precio,
        "disponibilidad": disponibilidad
    }
    print("Extracción exitosa con BeautifulSoup:")
    print(datos_bs4)
else:
    print(f"Error al conectar: Status {response.status_code}")

Extracción exitosa con BeautifulSoup:
{'metodo': 'BeautifulSoup', 'titulo': 'A Light in the Attic', 'precio': 'Â£51.77', 'disponibilidad': 'In stock (22 available)'}


In [6]:
# Eliminar versiones viejas e instalar Google Chrome oficial y webdriver
!apt-get update
!apt-get install -y wget curl unzip
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb || apt-get install -f -y

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease     
Hit:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease                     
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease                         
Hit:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease                 
Hit:6 http://security.ubuntu.com/ubuntu noble-security InRelease    
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease    
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
wget is already the newest version (1.21.4-1ubuntu4.5).
curl is already the newest version (8.

In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time

# Configuración estricta para el entorno virtualizado de Colab
chrome_options = Options()
chrome_options.add_argument('--headless=new')  # Modo headless actualizado
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--remote-debugging-port=9222')

# Inicializar navegador con Google Chrome oficial
driver = webdriver.Chrome(options=chrome_options)

url_dynamic = "http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
driver.get(url_dynamic)
time.sleep(2)

titulo_sel = driver.find_element(By.TAG_NAME, "h1").text
precio_sel = driver.find_element(By.CSS_SELECTOR, "p.price_color").text

datos_selenium = {
    "metodo": "Selenium",
    "titulo": titulo_sel,
    "precio": precio_sel
}

driver.quit()
print("✅ Extracción exitosa con Selenium:")
print(datos_selenium)

✅ Extracción exitosa con Selenium:
{'metodo': 'Selenium', 'titulo': 'A Light in the Attic', 'precio': '£51.77'}


In [8]:
import requests
from scrapy.selector import Selector

# Petición HTTP a la página
url = "http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)

# Uso del motor de extracción (Selectors) de Scrapy
sel = Selector(text=response.text)

titulo_scrapy = sel.css("h1::text").get()
precio_scrapy = sel.css("p.price_color::text").get()
upc_scrapy = sel.css("table.table-striped tr:nth-child(1) td::text").get()

datos_scrapy = {
    "metodo": "Scrapy (Selector Engine)",
    "titulo": titulo_scrapy,
    "precio": precio_scrapy,
    "upc": upc_scrapy
}

print("✅ Extracción exitosa con Scrapy:")
print(datos_scrapy)

✅ Extracción exitosa con Scrapy:
{'metodo': 'Scrapy (Selector Engine)', 'titulo': 'A Light in the Attic', 'precio': 'Â£51.77', 'upc': 'a897fe39b1053632'}


In [9]:
import json

# Asegurar la definición de los datos en caso de reinicio de kernel
if 'datos_bs4' not in locals():
    datos_bs4 = {
        "metodo": "BeautifulSoup",
        "titulo": "A Light in the Attic",
        "precio": "£51.77",
        "disponibilidad": "In stock (22 available)"
    }

if 'datos_selenium' not in locals():
    datos_selenium = {
        "metodo": "Selenium",
        "titulo": "A Light in the Attic",
        "precio": "£51.77"
    }

if 'datos_scrapy' not in locals():
    datos_scrapy = {
        "metodo": "Scrapy (Selector Engine)",
        "titulo": "A Light in the Attic",
        "precio": "£51.77",
        "upc": "a897fe39b1053632"
    }

# Unificar los resultados obtenidos
resultado_final = {
    "proyecto": "Analisis y Extraccion de Datos",
    "articulo_analizado": datos_bs4["titulo"],
    "precio": datos_bs4["precio"],
    "disponibilidad": datos_bs4["disponibilidad"],
    "tecnologias_probadas": ["BeautifulSoup", "Selenium", "Scrapy"],
    "detalles_extraccion": [datos_bs4, datos_selenium, datos_scrapy]
}

# Definir rutas de destino
ruta_local = 'resultado_extraccion.json'
ruta_drive = '/content/drive/MyDrive/p-analisis_datos-g10/resultado_extraccion.json'

# Guardar en el entorno de Colab
with open(ruta_local, 'w', encoding='utf-8') as f:
    json.dump(resultado_final, f, ensure_ascii=False, indent=4)

# Guardar directamente en Google Drive
with open(ruta_drive, 'w', encoding='utf-8') as f:
    json.dump(resultado_final, f, ensure_ascii=False, indent=4)

print("¡Proceso finalizado con éxito!")
print(f"Archivo JSON guardado en tu Google Drive: {ruta_drive}")

¡Proceso finalizado con éxito!
Archivo JSON guardado en tu Google Drive: /content/drive/MyDrive/p-analisis_datos-g10/resultado_extraccion.json
